# Logits Preprocessing and Data Engineering

In [24]:
def default_params(): 
    return {
        'current_model': 'M2', 
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'name': '/workspaces/CodeSmells/semeru-datasets/code_smells/codesmell_dataset.csv',
            'content_column': 'code', 
            'number_samples': 55,
        },
        'default_max_position_embeddings' : 16384,
        'output_path': '../data/raw_logits',
        'preprocessed_dataset_dir' : '../datax/code_smells/dataset_preprocessing',
        'cache_dir': '../datax/hugging_face_cache',
        'log_file': '../datax/code_smells/logit_extraction.log', 
        'callbacks_dir' : '../datax/code_smells/callbacks',
        'causal_models': {
            'M2': 'mistralai/Mistral-7B-v0.3', #https://huggingface.co/codellama/CodeLlama-13b-hf
        },
    }
params = default_params()


#### Imports

In [25]:
import pandas as pd
import os
import time
import numpy as np
import torch
import gc

In [26]:
from transformers import AutoTokenizer, MistralForCausalLM 
from datasets import load_dataset

RuntimeError: Failed to import transformers.models.auto.tokenization_auto because of the following error (look up to see its traceback):
cannot import name 'GGUF_CONFIG_MAPPING' from 'transformers.integrations' (/usr/local/lib/python3.11/dist-packages/transformers/integrations/__init__.py)

In [16]:
import logging
#logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)
logging.basicConfig(
    filename=params['log_file'],
    filemode='a',
    format='%(asctime)s : %(levelname)s : %(message)s', 
    level=logging.INFO
    )

In [17]:
import seaborn as sns
from scipy import stats
from statistics import NormalDist
import matplotlib.pyplot as plt

#### Dataset

In [18]:
df_dataset = pd.read_json(params['preprocessed_dataset_dir'] + '/' + params['current_model'] + '_q_' + params['quantization'] + '.json', )

#### Model Loading

In [21]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded LlamaTokenizer - " + model_name)
     model = None
     match params['quantization']:
               case 'int4':
                    model = MistralForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
               case 'int8':
                    model = MistralForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
               case 'float32':
                    model = MistralForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
               case 'float16':
                    model = MistralForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
               case _: 
                    model = MistralForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded MistralForCausalLM - " + model_name)

     return tokenizer, model

In [22]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

NameError: name 'AutoTokenizer' is not defined

#### Softmax Normalization and Data Engineering

In [9]:
def topk_tuple( logit_vocab_tensor, largest, tokenizer_fn):
    "Run topk for a token"
    topk = logit_vocab_tensor.topk( k=1 , largest=largest ) #TODO K number of elements can be extended
    return ( tokenizer.convert_tokens_to_string([tokenizer_fn.decode(topk.indices)]), topk.values.item())

def min_max_logits( logit_vocab_sample_tensor, tokenizer_fn ):
    "Compute min_max for a sample"
    max_cases = []
    min_cases = []
    for logit_vocab_tensor in logit_vocab_sample_tensor:
        max_cases.append( topk_tuple( logit_vocab_tensor = logit_vocab_tensor, largest = True, tokenizer_fn = tokenizer_fn) ) #TST Max Logit
        min_cases.append( topk_tuple( logit_vocab_tensor = logit_vocab_tensor, largest = False, tokenizer_fn = tokenizer_fn) ) #TST Min Logit
    return max_cases, min_cases

def actual_logit( 
                 logit_vocab_sample_tensor, 
                 tokenized_prompt, 
                 tokenizer_fn,
                 ):
    "Compute actual logits for a sample"
    actual_logits_prompt = []
    for token_pos, id_token in enumerate( tokenized_prompt[1:] ): #Eliminate the first token prediction since we do not use it
        actual_logits_prompt.append(
            (   tokenizer.convert_tokens_to_string([tokenizer_fn.decode( int(id_token))]), #retrieving the name of the token with the id
                logit_vocab_sample_tensor[token_pos][int(id_token)].item()) #retrieving the logit given the position in the sequence and the position in the vocab
            )
    return actual_logits_prompt

In [10]:
soft = torch.nn.Softmax( dim = 0 ) #Flattening normalization

In [11]:
out= np.load(params['callbacks_dir']+ '/'+ params['current_model'] + '_q_' + params['quantization'] +'/' + 'logits_tensor[0]_batch[0].npy')
print(out.shape) #<sample,tokens,voc_tokens>
out = out[0]


(1, 64, 32000)


In [12]:
max_case,min_case = min_max_logits(
    logit_vocab_sample_tensor = [ soft( torch.from_numpy(token) ) for token in out], ####### 
    tokenizer_fn= tokenizer
    )
print(max_case)
assert len(max_case) == len(min_case)

[('#', 0.2821720838546753), ('_', 0.2682041525840759), ('count', 0.8004263043403625), ('=', 0.3361130356788635), ('Float', 0.2915164828300476), ('->', 0.9826384782791138), ('Int', 0.6749789118766785), ('->', 0.8075503706932068), ('cal', 0.36109668016433716), ('culate', 0.9994254112243652), ('Dis', 0.9997397065162659), ('count', 0.998976469039917), ('price', 0.16062597930431366), ('=', 0.5149981379508972), (' ', 0.4238396883010864), ('|', 0.9920405745506287), ('amount', 0.8113887310028076), ('<', 0.2632333040237427), ('', 0.9682050347328186), ('1', 0.48404502868652344), ('0', 0.9185038208961487), ('0', 0.8114004731178284), ('=', 0.5917035341262817), ('', 0.7039767503738403), ('*', 0.4487527012825012), ('', 0.9975994229316711), ('|', 0.9990414977073669), ('amount', 0.5961533188819885), ('>=', 0.4537866413593292), ('', 0.9960363507270813), ('2', 0.4735748767852783), ('0', 0.9975233674049377), ('0', 0.9927055239677429), ('=', 0.9748342037200928), ('amount', 0.6801040172576904), ('(', 0.409

In [13]:
assert tokenizer.decode(df_dataset['input_ids'][0]) == df_dataset[params['dataset']['content_column']][0]
df_dataset[params['dataset']['content_column']][0]

2024-03-29 20:02:27.842895: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-03-29 20:02:27.842961: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-03-29 20:02:27.844016: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-03-29 20:02:27.850466: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


'calculateDiscount :: Int -> Int\ncalculateDiscount amount\n  | amount < 100 = amount\n  | amount < 500 = floor (fromIntegral amount * 0.9)\n  | otherwise = floor (fromIntegral amount * 0.8)'

In [14]:
input_ids_list = tokenizer.batch_encode_plus(df_dataset[params['dataset']['content_column']].tolist())
input_ids_list = [torch.tensor(  input_ids, dtype = torch.int) for input_ids in input_ids_list.input_ids]

actual_cases = actual_logit(
    logit_vocab_sample_tensor = [ soft( torch.from_numpy(token) ) for token in out] , #Out is a complete sequence
    tokenized_prompt = input_ids_list[0], ## SAMPLE ID
    tokenizer_fn = tokenizer
    )
actual_cases

[('calculate', 1.8452654160228121e-07),
 ('Dis', 0.0008257570443674922),
 ('count', 0.8004263043403625),
 ('::', 0.03860512375831604),
 ('Int', 0.23979492485523224),
 ('->', 0.9826384782791138),
 ('Int', 0.6749789118766785),
 ('\n', 0.18446189165115356),
 ('cal', 0.36109668016433716),
 ('culate', 0.9994254112243652),
 ('Dis', 0.9997397065162659),
 ('count', 0.998976469039917),
 ('amount', 0.03138299286365509),
 ('\n', 0.36234644055366516),
 ('', 0.40443065762519836),
 ('|', 0.9920405745506287),
 ('amount', 0.8113887310028076),
 ('<', 0.2632333040237427),
 ('', 0.9682050347328186),
 ('1', 0.48404502868652344),
 ('0', 0.9185038208961487),
 ('0', 0.8114004731178284),
 ('=', 0.5917035341262817),
 ('amount', 0.2432878613471985),
 ('\n', 0.3838385045528412),
 ('', 0.9975994229316711),
 ('|', 0.9990414977073669),
 ('amount', 0.5961533188819885),
 ('<', 0.3268502652645111),
 ('', 0.9960363507270813),
 ('5', 0.21014757454395294),
 ('0', 0.9975233674049377),
 ('0', 0.9927055239677429),
 ('=', 0.

#### Processing all the Batches

In [15]:
def batching_logits(tokenizer,tf_input_ids,size=10000):
    max_logit_token_prompt = []
    min_logit_token_prompt = []
    actual_logit_token_prompt = []

    
    soft = torch.nn.Softmax( dim = 0 )                          #Flattening normalization
    
    for file in range( size ):
        out = np.load(params['callbacks_dir']+ '/'+ params['current_model'] + '_q_' + params['quantization'] +'/'+ f'logits_tensor[{file}]_batch[{file}].npy') #<sample,tokens,voc_tokens>
        out = out[0]  ##### #<tokens,voc_tokens>
        next_tokens_distribution = [ soft( torch.from_numpy(token) ) for token in out]  #Flattening normalization
        
        max_cases,min_cases = min_max_logits(
            logit_vocab_sample_tensor = next_tokens_distribution,
            tokenizer_fn= tokenizer
            )

        actual_cases = actual_logit(
            logit_vocab_sample_tensor = next_tokens_distribution,
            tokenized_prompt = tf_input_ids[ file ],
            tokenizer_fn = tokenizer
            )
        
        max_logit_token_prompt.append( max_cases )
        min_logit_token_prompt.append( min_cases )
        actual_logit_token_prompt.append( actual_cases )
        
        logging.info(file)
    return max_logit_token_prompt,min_logit_token_prompt,actual_logit_token_prompt

In [16]:
input_ids_list = tokenizer.batch_encode_plus(df_dataset[params['dataset']['content_column']].tolist())
input_ids_list = [torch.tensor(  input_ids, dtype = torch.int) for input_ids in input_ids_list.input_ids]

In [17]:
max_logit_token_prompt, min_logit_token_prompt, actual_logit_token_prompt = batching_logits(
    tokenizer=tokenizer , tf_input_ids=input_ids_list, 
    size = params['dataset']['number_samples']
) #<---WARNING TIME Consuming

#### Saving results

In [18]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [19]:
dataframe_to_save = df_dataset.copy()
dataframe_to_save['max_prob'] = max_logit_token_prompt
dataframe_to_save['min_prob'] = min_logit_token_prompt
dataframe_to_save['actual_prob'] = actual_logit_token_prompt
dataframe_to_save.shape

(55, 12)

In [20]:
dataframe_to_save.head(5)

,problem_type,problem,full_solution,entry_point,function,signature,context,unit_test_template,input_ids,max_prob,min_prob,actual_prob
0,function,The function `calculateDiscount` takes a singl...,calculateDiscount :: Int -> Int\ncalculateDisc...,calculateDiscount,calculateDiscount amount\n | amount < 100 = a...,calculateDiscount :: Int -> Int,None,import Test.Hspec\n<FILL SIGNATURE>\n<FILL FUN...,"[13911, 3278, 2114, 6210, 4666, 3193, 4666, 13...","[(#, 0.2821720838546753), (_, 0.26820415258407...","[(–,, 1.963395279691582e-10), (même, 1.5962491...","[(calculate, 1.8452654160228121e-07), (Dis, 0...."
1,function,The function `calculateThemeChange` takes two ...,calculateThemeChange :: String -> String -> St...,calculateThemeChange,calculateThemeChange current desired\n | curr...,calculateThemeChange :: String -> String -> St...,None,import Test.Hspec\n<FILL SIGNATURE>\n<FILL FUN...,"[13911, 10438, 5500, 6210, 1677, 3193, 1677, 3...","[(#, 0.2821975648403168), (_, 0.26990848779678...","[(–,, 1.963572499041888e-10), (même, 1.5938916...","[(calculate, 1.847235040486339e-07), (Theme, 2..."
2,function,The function `updatePath` takes two strings as...,import Data.List (isInfixOf)\n\nupdatePath :: ...,updatePath,updatePath existingPath newDir\n | newDir `is...,updatePath :: String -> String -> String,import Data.List (isInfixOf)\n\n,import Data.List (isInfixOf)\nimport Test.Hspe...,"[726, 5284, 28723, 1245, 325, 278, 657, 7192, ...","[(#, 0.28298500180244446), ({, 0.3113600313663...","[(–,, 1.9537282902604147e-10), (/******/, 2.43...","[(import, 0.023048054426908493), (Data, 0.0003..."
3,function,Design a function named `calculateDiscount` th...,calculateDiscount :: Float -> Int -> Bool -> F...,calculateDiscount,calculateDiscount price discountPercentage onC...,calculateDiscount :: Float -> Int -> Bool -> F...,None,import Test.Hspec\n<FILL SIGNATURE>\n<FILL FUN...,"[13911, 3278, 2114, 6210, 27914, 3193, 4666, 3...","[(#, 0.28301000595092773), (_, 0.2698296010494...","[(–,, 1.938695731729112e-10), (même, 1.6059231...","[(calculate, 1.8184967132128804e-07), (Dis, 0...."
4,function,The function `filterAndCount` takes a list of ...,import Data.List (filter)\n\nfilterAndCount ::...,filterAndCount,filterAndCount strs char threshold = length $ ...,filterAndCount :: [String] -> Char -> Int -> Int,import Data.List (filter)\n\n,import Data.List (filter)\nimport Test.Hspec\n...,"[726, 5284, 28723, 1245, 325, 4650, 28731, 13,...","[(#, 0.28298500180244446), ({, 0.3113600313663...","[(–,, 1.9537282902604147e-10), (/******/, 2.43...","[(import, 0.023048054426908493), (Data, 0.0003..."


In [21]:
create_folder(params['output_path'] + '/' + params['current_model'] + '_q_' + params['quantization'])
dataframe_to_save.to_csv( params['output_path'] + '/' + params['current_model'] + '_q_' + params['quantization'] + '/' + 'raw_logits.csv')

#### Loss Retrieval

In [22]:
def batching_loss( size = dataframe_to_save.shape[0] ):
    output_loss = []
    for current_batch in range(size):
        out = np.load(params['callbacks_dir']+ '/'+ params['current_model'] +  '_q_' + params['quantization'] +'/' + f'_loss_batch[{current_batch}].npy') 
        output_loss.append( out.item() ) #.item() for numpy library
        logging.info(current_batch)
    return output_loss

In [23]:
output_loss = batching_loss() #[WAENING!] Takes Time

In [24]:
output_loss

[0.9163909554481506,
 1.6981748342514038,
 1.3668224811553955,
 0.8484765887260437,
 0.9623405933380127,
 0.8796409368515015,
 1.9571266174316406,
 0.7909044027328491,
 1.0484223365783691,
 1.318931221961975,
 1.0876078605651855,
 1.1862108707427979,
 0.8415679335594177,
 0.9844745993614197,
 1.482188105583191,
 1.41868257522583,
 0.9519271850585938,
 0.8719722628593445,
 1.0741350650787354,
 1.264746069908142,
 1.3716787099838257,
 1.301838755607605,
 0.8080506920814514,
 1.5023620128631592,
 1.4263558387756348,
 1.3295791149139404,
 1.367255687713623,
 1.7442253828048706,
 0.9661988615989685,
 1.369888424873352,
 1.1228433847427368,
 1.4408659934997559,
 0.891493558883667,
 1.0229685306549072,
 0.8160669803619385,
 0.9662980437278748,
 0.910210907459259,
 1.1756296157836914,
 0.9944772124290466,
 0.9764159917831421,
 1.034815788269043,
 0.9661810994148254,
 1.2697499990463257,
 0.9980368614196777,
 1.7931010723114014,
 1.1407517194747925,
 0.9975525736808777,
 1.5442391633987427,
 0.

In [25]:
dataframe_to_save['loss'] = output_loss
dataframe_to_save.head(5)

,problem_type,problem,full_solution,entry_point,function,signature,context,unit_test_template,input_ids,max_prob,min_prob,actual_prob,loss
0,function,The function `calculateDiscount` takes a singl...,calculateDiscount :: Int -> Int\ncalculateDisc...,calculateDiscount,calculateDiscount amount\n | amount < 100 = a...,calculateDiscount :: Int -> Int,None,import Test.Hspec\n<FILL SIGNATURE>\n<FILL FUN...,"[13911, 3278, 2114, 6210, 4666, 3193, 4666, 13...","[(#, 0.2821720838546753), (_, 0.26820415258407...","[(–,, 1.963395279691582e-10), (même, 1.5962491...","[(calculate, 1.8452654160228121e-07), (Dis, 0....",0.916391
1,function,The function `calculateThemeChange` takes two ...,calculateThemeChange :: String -> String -> St...,calculateThemeChange,calculateThemeChange current desired\n | curr...,calculateThemeChange :: String -> String -> St...,None,import Test.Hspec\n<FILL SIGNATURE>\n<FILL FUN...,"[13911, 10438, 5500, 6210, 1677, 3193, 1677, 3...","[(#, 0.2821975648403168), (_, 0.26990848779678...","[(–,, 1.963572499041888e-10), (même, 1.5938916...","[(calculate, 1.847235040486339e-07), (Theme, 2...",1.698175
2,function,The function `updatePath` takes two strings as...,import Data.List (isInfixOf)\n\nupdatePath :: ...,updatePath,updatePath existingPath newDir\n | newDir `is...,updatePath :: String -> String -> String,import Data.List (isInfixOf)\n\n,import Data.List (isInfixOf)\nimport Test.Hspe...,"[726, 5284, 28723, 1245, 325, 278, 657, 7192, ...","[(#, 0.28298500180244446), ({, 0.3113600313663...","[(–,, 1.9537282902604147e-10), (/******/, 2.43...","[(import, 0.023048054426908493), (Data, 0.0003...",1.366822
3,function,Design a function named `calculateDiscount` th...,calculateDiscount :: Float -> Int -> Bool -> F...,calculateDiscount,calculateDiscount price discountPercentage onC...,calculateDiscount :: Float -> Int -> Bool -> F...,None,import Test.Hspec\n<FILL SIGNATURE>\n<FILL FUN...,"[13911, 3278, 2114, 6210, 27914, 3193, 4666, 3...","[(#, 0.28301000595092773), (_, 0.2698296010494...","[(–,, 1.938695731729112e-10), (même, 1.6059231...","[(calculate, 1.8184967132128804e-07), (Dis, 0....",0.848477
4,function,The function `filterAndCount` takes a list of ...,import Data.List (filter)\n\nfilterAndCount ::...,filterAndCount,filterAndCount strs char threshold = length $ ...,filterAndCount :: [String] -> Char -> Int -> Int,import Data.List (filter)\n\n,import Data.List (filter)\nimport Test.Hspec\n...,"[726, 5284, 28723, 1245, 325, 4650, 28731, 13,...","[(#, 0.28298500180244446), ({, 0.3113600313663...","[(–,, 1.9537282902604147e-10), (/******/, 2.43...","[(import, 0.023048054426908493), (Data, 0.0003...",0.962341


In [26]:
## Saving CheckPoint 2
dataframe_to_save.to_csv( params['output_path'] + '/' + params['current_model'] + '_q_' + params['quantization'] + '/' + 'raw_logits.csv')

In [27]:
torch.cuda.empty_cache()
gc.collect()

334